<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L4/4a_pretrained_models_ADL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Deep Learning
## Pre-Trained Models

Pre-trained models have been trained with large amounts of data and are useful when we don't have enough data for our problem. When starting a new deep learning project, it is a good idea to first check whether an existing pretrained model can provide a useful baseline. PyTorch and TorchVision provide a range of pretrained computer-vision models, and many additional models are available through open-source model repositories and GitHub.

In [ ]:
%%bash

mkdir data
wget -q https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/iti121/doggy_door_images.zip
wget -q https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/iti121/imagenet_class_index.json -O data/imagenet_class_index.json

unzip doggy_door_images.zip -d data

## 3a.1 Objectives

* Use TorchVision to load a very well-trained pretrained model
* Preprocess our own images to work with the pretrained model
* Use the pretrained model to perform accurate inference on your own images

In [ ]:
import torch
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

## 3a.2 Using a pretrained model

The [ImageNet challenge](https://en.wikipedia.org/wiki/ImageNet#History_of_the_ImageNet_challenge) has produced many state-of-the-art models that can be used for image classification. They are trained on millions of images, and can accurately classify images into 1000 different categories. Many of those categories are animals, including breeds of dogs and cats.

## 3a.3 Loading the Model

We will start by downloading the model. Trained ImageNet models are available to download directly within the TorchVision library. We can see the available models and their details [here](https://pytorch.org/vision/stable/models.html). Any of these models would work for our exercise. We will pick a commonly used one called [VGG16](https://pytorch.org/vision/stable/models/vgg.html) and use the [default weights](https://pytorch.org/vision/stable/models/generated/torchvision.models.vgg19.html#torchvision.models.VGG19_Weights).

In [ ]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

# load the VGG16 network *pre-trained* on the ImageNet dataset
weights = VGG16_Weights.DEFAULT
model = vgg16(weights=weights)

Now that it is loaded, let us look at the model. It looks a lot like our convolutional model from the sign language exercise.

In [ ]:
model.to(device)

### 3a.3.1 Input dimensions

As with our earlier exercises, our images should match the input dimensions that the model expects. Because PyTorch models are dynamically built, the model itself does not know what its input shape should be. Thankfully, the pretrained `weights` come with their own transforms.

In [ ]:
pre_trans = weights.transforms()
pre_trans

This is equivalent to the following:

```Python
pre_trans=transforms.Compose([
    # 1. Scale the smaller edge to 256 while keeping aspect ratio
    v2.Resize(size=256, interpolation=v2.InterpolationMode.BILINEAR),
    
    # 2. Cut out the center 224x224 square
    v2.CenterCrop(size=224),
    
    # 3. Convert integer pixels to float32 and scale from [0, 255] to [0.0, 1.0]
    v2.ToDtype(torch.float32, scale=True),
    
    # 4. Standardise the data using ImageNet channel averages
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
```

One new Transform here is [Normalize](https://pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html) which will recolor an image based on a provided [mean](https://en.wikipedia.org/wiki/Mean) and [standard deviation](https://en.wikipedia.org/wiki/Standard_deviation). The other new Transform is [CenterCrop](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.CenterCrop.html#torchvision.transforms.v2.CenterCrop), which removes the edges of the image. We can apply these transforms to any image tensor we'd like to use with the VGG16 model.

### 3a.3.2 Output dimensions

We can also see that the model will return a prediction of shape 1000. This means that we have 1000 possible categories that the image will be placed in. Though the full ImageNet dataset has over 20,000 categories, the competition and resulting pre-trained models just use a subset of 1000 of these categories. We can take a look at all of these [possible categories here](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a).

Many of the categories are animals, including many types of dogs and cats. The dogs are categories 151 through 268. The cats are categories 281 through 285.

## 3a.4 Loading an Image
We will start by loading in an image and displaying it, as we have done in previous exercises.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(image_path):
    image = mpimg.imread(image_path)
    plt.imshow(image)

In [ ]:
show_image("data/doggy_door_images/happy_dog.jpg")

### 3a.4.1 Preprocessing the Image

Next, we will preprocess the image so that it is ready to be sent into the model. Remember that in this case, the final shape of the image should be (1, 3, 224, 224). We will use the Transforms provided by `weights` to so this.

In [ ]:
def load_and_process_image(file_path):
    # Print image's original shape, for reference
    print('Original image shape: ', mpimg.imread(file_path).shape)

    image = tv_io.read_image(file_path).to(device)
    image = pre_trans(image)  # weights.transforms()
    image = image.unsqueeze(0)  # Turn into a batch
    return image

Let's see if it worked on the happy dog photo:

In [ ]:
processed_image = load_and_process_image("data/doggy_door_images/happy_dog.jpg")
print("Processed image shape: ", processed_image.shape)

The shape is correct, but how does it look? Let's print an image to verify:

In [ ]:
import torchvision.transforms.functional as F

plot_image = F.to_pil_image(torch.squeeze(processed_image))
plt.imshow(plot_image, cmap='gray')

The colours look a bit strange, but if we look carefully, we can still see our happy dog. The strange coloring is due to the `Normalize` Transform, and the photo appears a little zoomed in because of `CenterCrop`.

## 3a.5 Make a Prediction

Now that we have our image in the right format, we can pass it into our model and get a prediction. We are expecting an output of an array of 1000 elements, which is going to be difficult to read. We have a list of all the classes in [json](https://www.json.org/json-en.html) format, which is similar to a combination of python lists and dictionaries.

In [ ]:
vgg_classes = json.load(open("data/imagenet_class_index.json"))

This file uses strings of numbers for each of the classes:

In [ ]:
vgg_classes["0"]

Let's create a function to make a prediction from the VGG model human readable. This is similar to our `predict_letter` function from the previous lesson. We will use the [torch.topk](https://pytorch.org/docs/stable/generated/torch.topk.html) function to give us the top `3` predictions.

In [ ]:
def readable_prediction(image_path):
    # Show image
    show_image(image_path)
    # Load and pre-process image
    image = load_and_process_image(image_path)
    # Make predictions
    output = model(image)[0]  # Unbatch
    predictions = torch.topk(output, 3)
    indices = predictions.indices.tolist()
    # Print predictions in readable form
    out_str = "Top results: "
    pred_classes = [vgg_classes[str(idx)][1] for idx in indices]
    out_str += ", ".join(pred_classes)
    print(out_str)

    return predictions

Try it out on a few animals to see the results! Also feel free to upload your own images and categorize them just to see how well it works.

In [ ]:
readable_prediction("data/doggy_door_images/happy_dog.jpg")

In [ ]:
readable_prediction("data/doggy_door_images/brown_bear.jpg")

In [ ]:
readable_prediction("data/doggy_door_images/sleepy_cat.jpg")

## 3a.6 Cats and Dogs

Let's say we want to use this pretrained model to build a model that identifies all dogs and cats. The dogs are categories 151 through 268 and cats are categories 281 through 285.

#### Exercise

Take a look at the function below and fill in the missing item.

**Hint**: The first dimension is the batch dimension.

In [ ]:
def cats_and_dogs(image_path):
    show_image(image_path)
    image = load_and_process_image(image_path)
    idx = model(image).argmax(dim=FIXME).item()
    print("Predicted index:", idx)
    if 151 <= idx <= 268:
        print("This is a dog!")
    elif 281 <= idx <= 285:
        print("This is a cat!")
    else:
        print("This is not a cat or a dog!")

### Solution

In [ ]:
# SOLUTION
import numpy as np

def cats_and_dogs(image_path):
    show_image(image_path)
    image = load_and_process_image(image_path)
    idx = model(image).argmax(dim=1).item()
    print("Predicted index:", idx)
    if 151 <= idx <= 268:
        print("This is a dog!")
    elif 281 <= idx <= 285:
        print("This is a cat!")
    else:
        print("This is not a cat or a dog!")

In [ ]:
cats_and_dogs("data/doggy_door_images/brown_bear.jpg")

In [ ]:
cats_and_dogs("data/doggy_door_images/happy_dog.jpg")

In [ ]:
cats_and_dogs("data/doggy_door_images/sleepy_cat.jpg")

## Next

Using pretrained models is incredibly powerful, but sometimes they are not a perfect fit for your data. In the next section you will learn about another powerful technique, *transfer learning*, which allows you to tailer pretrained models to make good predictions for your data.